# Long-run noise hypothesis in EmbryoMaker Flow

This notebook tests one simulator hypothesis on the bundled differential adhesion example. Both conditions use the same starting state and random seed, then advance for exactly 1,000 solver iterations.

Hypothesis: raising `prop_noise` from 0.50 to 0.75 increases mean node displacement from the initial state after 1,000 iterations. The readout is mean Euclidean displacement across corresponding nodes. This is a simulator behavior test, not biological evidence.

The notebook starts the included Linux simulator server, submits each input through its local API, advances in 25-iteration chunks, and saves both snapshots plus a JSON result in the Colab runtime.

In [ ]:
from pathlib import Path
import atexit, json, math, re, subprocess, tarfile, time, urllib.error, urllib.request
import matplotlib.pyplot as plt

if 'server' in globals() and server.poll() is None:
    server.terminate()
    server.wait(timeout=10)
WORK = Path('/content/embryomaker_hypothesis')
WORK.mkdir(parents=True, exist_ok=True)
ARCHIVE_URL = 'https://godofecht.github.io/embryo-maker-flow/downloads/EmbryoMaker-Flow-linux-x86_64-v0.1.0.tar.gz'
ARCHIVE = WORK / 'embryomaker-linux.tar.gz'
if not ARCHIVE.exists():
    urllib.request.urlretrieve(ARCHIVE_URL, ARCHIVE)
with tarfile.open(ARCHIVE, 'r:gz') as archive:
    root = WORK.resolve()
    members = []
    for member in archive.getmembers():
        target = (WORK / member.name).resolve()
        if target != root and root not in target.parents:
            raise ValueError(f'Unsafe archive path: {member.name}')
        if '/._' not in member.name and not Path(member.name).name.startswith('._'):
            members.append(member)
    archive.extractall(WORK, members=members, filter='data')
PROJECT = WORK / 'EmbryoMaker-Flow'
BINARY = PROJECT / 'EmbryoMaker'
INPUT = PROJECT / 'reference/examples/differential_adhesion/differential_adhesion.txt'
assert BINARY.is_file() and INPUT.is_file(), 'Release archive is missing the simulator or example input.'
BINARY.chmod(0o755)
BASE_URL = 'http://127.0.0.1:8765'
if 'server' not in globals() or server.poll() is not None:
    server = subprocess.Popen([str(BINARY)], cwd=PROJECT, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
atexit.register(lambda: server.terminate() if server.poll() is None else None)
for attempt in range(60):
    try:
        with urllib.request.urlopen(BASE_URL + '/api/examples', timeout=2) as response:
            examples = json.load(response)
        break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError('Simulator server exited during startup.')
        time.sleep(1)
else:
    raise TimeoutError('Simulator API did not become ready.')
assert any(item['id'] == 'differential_adhesion' for item in examples['examples'])
print('Simulator API ready.')
print('Example: differential adhesion. Nodes in input: 168.')
print('Iterations per condition: 1,000')

In [ ]:
def change_parameter(source: Path, destination: Path, row_number: int, value: float):
    lines = source.read_text(encoding='utf-8', errors='replace').splitlines(keepends=True)
    section = next(i for i, line in enumerate(lines) if line.strip() == 'parameters')
    for i in range(section + 1, len(lines)):
        fields = lines[i].split()
        if not fields:
            continue
        if not fields[0].isdigit():
            break
        if int(fields[0]) == row_number:
            match = re.match(r'^(\s*\d+\s+)\S+(.*)$', lines[i], re.DOTALL)
            if not match:
                raise ValueError(f'Malformed parameter row {row_number}')
            lines[i] = match.group(1) + format(value, '.17g') + match.group(2)
            destination.write_text(''.join(lines), encoding='utf-8')
            return
    raise ValueError(f'Parameter row {row_number} was not found')

VARIANT = WORK / 'differential_adhesion_prop_noise_0.75.txt'
change_parameter(INPUT, VARIANT, row_number=21, value=0.75)
print('Control prop_noise: 0.50')
print('Variant prop_noise: 0.75')
print('Changed input row: 21 (M(NOI), proportion of nodes subject to noise)')

In [ ]:
ITERATIONS = 1000
CHUNK_SIZE = 25
RUNS = WORK / 'results'
RUNS.mkdir(exist_ok=True)

def post_json(path: str, payload: dict, timeout: int = 600):
    request = urllib.request.Request(BASE_URL + path, data=json.dumps(payload).encode('utf-8'), headers={'Content-Type': 'application/json'}, method='POST')
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.load(response)
    except urllib.error.HTTPError as error:
        detail = error.read().decode('utf-8', errors='replace')
        raise RuntimeError(f'API returned HTTP {error.code}: {detail}') from error

def run_condition(label: str, input_path: Path):
    input_text = input_path.read_text(encoding='utf-8', errors='replace')
    model = post_json('/api/session', {'inputText': input_text, 'fileName': input_path.name})
    session = model['session']
    if model['iterations'] != 0:
        raise RuntimeError(f'{label} did not begin at iteration zero: {model["iterations"]}')
    started = time.time()
    snapshot = None
    for completed in range(0, ITERATIONS, CHUNK_SIZE):
        count = min(CHUNK_SIZE, ITERATIONS - completed)
        final_chunk = completed + count == ITERATIONS
        data = post_json(f'/api/session/{session}/advance', {'steps': count, 'includeSnapshot': final_chunk})
        expected = completed + count
        if data.get('iterations') != expected:
            raise RuntimeError(f'{label} stopped at {data.get("iterations")} iterations, expected {expected}')
        if final_chunk:
            snapshot = data.get('snapshot')
            if not snapshot:
                raise RuntimeError(f'{label} final response has no snapshot')
        print(f'{label}: {expected}/{ITERATIONS} iterations', flush=True)
    snapshot_path = RUNS / f'{label}_{ITERATIONS}.dat'
    snapshot_path.write_text(snapshot, encoding='utf-8')
    return {'label': label, 'snapshot': snapshot_path, 'wall_seconds': time.time() - started}

def node_positions(text: str, marker: str):
    lines = text.splitlines()
    cursor = lines.index(marker) + 1
    while cursor < len(lines) and not lines[cursor].strip():
        cursor += 1
    if cursor >= len(lines) or lines[cursor].strip() != 'node parameters':
        raise ValueError(f'Malformed snapshot section: {marker}')
    cursor += 1
    positions = []
    while cursor < len(lines) and lines[cursor].strip():
        fields = lines[cursor].split()
        positions.append(tuple(float(x) for x in fields[:3]))
        cursor += 1
    return positions

def mean_displacement(snapshot: Path):
    text = snapshot.read_text(encoding='utf-8', errors='replace')
    final = node_positions(text, 'node properties')
    initial = node_positions(text, 'node properties at time 0 (nodeo)')
    if len(final) != len(initial):
        raise ValueError(f'Node counts changed: initial {len(initial)}, final {len(final)}')
    ds = [math.dist(a, b) for a, b in zip(initial, final)]
    return {'nodes': len(ds), 'moved_nodes': sum(d > 0 for d in ds), 'mean_displacement': sum(ds) / len(ds), 'max_displacement': max(ds)}

control_run = run_condition('control', INPUT)
variant_run = run_condition('higher_noise', VARIANT)
control_metrics = mean_displacement(control_run['snapshot'])
variant_metrics = mean_displacement(variant_run['snapshot'])
print(json.dumps({'control': control_metrics, 'variant': variant_metrics}, indent=2))

In [ ]:
delta = variant_metrics['mean_displacement'] - control_metrics['mean_displacement']
ratio = variant_metrics['mean_displacement'] / control_metrics['mean_displacement'] if control_metrics['mean_displacement'] else float('nan')
result = {
    'hypothesis': 'Increasing prop_noise from 0.50 to 0.75 raises mean node displacement after 1000 iterations.',
    'iterations_per_condition': ITERATIONS,
    'input': 'differential_adhesion.txt',
    'control': {'prop_noise': 0.50, **control_metrics},
    'variant': {'prop_noise': 0.75, **variant_metrics},
    'difference_variant_minus_control': delta,
    'ratio_variant_over_control': ratio,
    'wall_seconds': {'control': control_run['wall_seconds'], 'variant': variant_run['wall_seconds']},
    'interpretation': 'Simulator output for this starting state and seed. It does not establish biological validity.'
}
(RUNS / 'hypothesis_result.json').write_text(json.dumps(result, indent=2) + '\n')

plt.figure(figsize=(6, 4))
plt.bar(['Control\nprop_noise 0.50', 'Variant\nprop_noise 0.75'], [control_metrics['mean_displacement'], variant_metrics['mean_displacement']], color=['#456b82', '#d47c52'])
plt.ylabel('Mean node displacement after 1,000 iterations')
plt.title('Differential adhesion example')
plt.tight_layout()
plt.show()
print(json.dumps(result, indent=2))
print('Snapshots and JSON saved under', RUNS)